In [9]:
import os
from datetime import datetime
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F



In [ ]:
df = pd.read_csv(f"BTCUSDT-15m-data.csv")
print(df)

**Check missing values**

In [ ]:
print(df.isnull())
print(f"Counts how many missing values there are in each column: {df.isnull().sum()}")
print(f"Total missing values: {df.isnull().sum().sum()}")

**Split data**

In [12]:
train_end_idx = 240_000

df_train = df.iloc[:train_end_idx].copy()
df_test = df.iloc[train_end_idx:].copy()

df_test.reset_index(drop=True, inplace=True)

print(f"Length of df_train: {len(df_train)}")
print(f"Length of df_test: {len(df_test)}")

Length of df_train: 240000
Length of df_test: 36000


**Building Meaningful Features**

In [13]:
def build_features(opens, closes):
    feature1 = (closes - opens) / opens

    # Stack features
    features = np.stack([
        feature1,
        # ...
    ], axis=-1) # shape: (time_steps, num_features)

    num_features = features.shape[-1]
    return features, num_features

**Preprocess data**

In [14]:
def preprocess_data(seq_len, df):
    m = len(df)

    opens = np.array(df['open'].values)
    closes = np.array(df['close'].values)

    # Build features
    features, num_features = build_features(opens, closes)

    # Calculate number of samples
    num_samples = m - seq_len

    # Create storage for inputs & targets
    X = np.zeros([num_samples, seq_len, num_features], dtype=np.float32)
    Y = np.zeros([num_samples, num_features], dtype=np.float32)

    # Create samples (X, Y)
    for i in range(num_samples):
        X[i] = features[i : i+seq_len]
        Y[i] = features[i+seq_len : i+seq_len+1]

    return X, Y, num_features

In [15]:
# Sequence Length
seq_len = 96

# Preprocess data
X_train, Y_train, num_features = preprocess_data(seq_len, df_train)
X_test, Y_test, num_features = preprocess_data(seq_len, df_test)


In [ ]:
m_train = X_train.shape[0]
m_test = X_test.shape[0]
print(f"m_train: {m_train}")
print(f"m_test: {m_test}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

**Transform data to Torch Tensor**

In [17]:
device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
print("my deive: ", device)

X_train = torch.from_numpy(X_train.astype(np.float32)).to(device, dtype=torch.float32)
Y_train = torch.from_numpy(Y_train.astype(np.float32)).to(device, dtype=torch.float32)

X_test = torch.from_numpy(X_test.astype(np.float32)).to(device, dtype=torch.float32)
Y_test = torch.from_numpy(Y_test.astype(np.float32)).to(device, dtype=torch.float32)

Y_pred_test = torch.zeros([m_test, num_features], device=device, dtype=torch.float32)

my deive:  cpu


**Build model**

In [18]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # Work station 1:
        self.fc1 = nn.Linear(in_features=seq_len*num_features,
                             out_features=20)
        self.act1 = nn.ReLU()

        # Work station 2:
        self.fc2 = nn.Linear(in_features=20,
                             out_features=40)
        self.act2 = nn.ReLU()

        # Head manager:
        self.fc_out = nn.Linear(in_features=40,
                                out_features=num_features)

    def forward(self, x):
        # x shape: (batch_size, seq_len, num_features)
        x = x.reshape(-1, seq_len*num_features)

        # Pass to Workstation 1
        x = self.fc1(x)
        x = self.act1(x)

        # Pass to Workstation 2
        x = self.fc2(x)
        x = self.act2(x)

        # Get final output
        x = self.fc_out(x)

        return x



**Initialization**

In [19]:
# Initialize model
model = Model()

# Transfer model to device
model.to(device)

# Initialize criterion
criterion = nn.MSELoss()

# Choose learning rate
lr = 1e-4

# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Choose batch size
batch_size = 32

# Lists of train/test losses
train_losses = []
test_losses = []

# Set epoch
num_epochs = 200



In [20]:
def create_minibatches(X, Y, batch_size, shuffle=True):
    mini_batches = []
    m = len(X)

    if shuffle:
        permutation = list(np.random.permutation(m))
        X = X[permutation]
        Y = Y[permutation]

    num_batches = m // batch_size

    for k in range(num_batches):
        mb_X = X[k * batch_size : k * batch_size + batch_size]
        mb_Y = Y[k * batch_size : k * batch_size + batch_size]
        mb_pair = (mb_X, mb_Y)
        mini_batches.append(mb_pair)

    return mini_batches

**Train model**

In [ ]:
for epoch in range(num_epochs):
    # Track duration
    dt0_epoch = datetime.now()

    # Create mini batches
    mini_batches = create_minibatches(X_train, Y_train, batch_size, shuffle=True)

    train_losses_batch = []

    for X_train_batch, Y_train_batch in mini_batches:
        # Set train mode
        model.train()

        # Reset the gradients
        optimizer.zero_grad()

        # Forward propagation
        Y_hat_train = model(X_train_batch) # Shape: (batch_size, num_features)

        # Calculate loss (train)
        train_loss = criterion(Y_hat_train, Y_train_batch)

        # Backprop
        train_loss.backward()

        # Update optimizer
        optimizer.step()

        # Keep track of training losses
        train_losses_batch.append(train_loss.item())

    # Predict on test data
    with torch.no_grad():

        # Set evaluation mode
        model.eval()

        # Forward propagation
        Y_pred_test = model(X_test)

        # Calculate test loss
        test_loss = criterion(Y_pred_test, Y_test)

    # Convert 'Y_pred_test' to numpy
    Y_pred_test_cpu = Y_pred_test.detach().cpu().numpy()

    train_losses.append(np.mean(train_losses_batch))
    test_losses.append(test_loss.item())

    print(f'Epoch: {epoch}, train_losses: {train_losses[-1]}, test_losses: {test_losses[-1]}, Duration: {datetime.now() - dt0_epoch}')

    # Plot Train Loss and Test Loss
    plt.title(f"Losses", fontsize=10)
    plt.plot(train_losses, label="Train Loss", color='deepskyblue')
    plt.plot(test_losses, label="Test Loss", color='orange')
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid()
    # Display
    plt.tight_layout()
    plt.show()
    plt.close()
